# Analisis Kinerja RAM dan Response Time Aplikasi Android
## BAB IV: Hasil dan Pembahasan - Pengujian Performa

Notebook ini digunakan untuk mengolah data mentah hasil benchmark performa aplikasi Android pada dua perangkat:
1. **Redmi Note 10 Pro** (Xiaomi M2101K6G, Android 12, API 31)
2. **Redmi 9A** (Xiaomi M2006C3LG, Android 10, API 29)

Setiap perangkat diuji sebanyak **3 kali sesi pengujian (runs)**. Notebook ini secara otomatis mengekstrak data dari file log mentah di dalam folder `dataset`, menghitung rata-rata serta rentang nilai minimum-maksimum, membuat tabel markdown, dan menghasilkan visualisasi grafik yang siap digunakan untuk penulisan **Skripsi S1**.

*Catatan: Notebook ini dikonfigurasi untuk memuat data dari folder `dataset` dan menyimpan seluruh hasil analisis langsung ke folder `output`.*

In [ ]:
import sys
import subprocess

# Mengotomatiskan instalasi dependensi jika belum ada di lingkungan runtime
required_libraries = {'pandas', 'numpy', 'matplotlib', 'seaborn', 'tabulate'}
installed_libraries = set()

for lib in required_libraries:
    try:
        __import__(lib)
        installed_libraries.add(lib)
    except ImportError:
        pass

missing_libraries = required_libraries - installed_libraries
if missing_libraries:
    print(f"Menginstal dependensi yang belum ada: {missing_libraries}...")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_libraries])
        print("Semua dependensi berhasil diinstal!")
    except Exception as e:
        print(f"Gagal menginstal dependensi: {e}")
else:
    print("Semua dependensi yang diperlukan sudah terpasang.")

In [ ]:
import os
import re
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Konfigurasi gaya visualisasi agar terlihat premium dan rapi
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans', 'Arial', 'Helvetica', 'Microsoft YaHei'],
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.titlesize': 15,
    'legend.fontsize': 10,
    'figure.dpi': 150
})

# Warna kustom profesional (Teal vs Amber)
COLORS = ['#008080', '#FF8C00']

# Membuat folder output jika belum ada
os.makedirs('output', exist_ok=True)
print("Library berhasil diimpor dan konfigurasi folder output siap.")

### 1. Fungsi Parser Log Kinerja
Fungsi berikut membaca log benchmark secara dinamis dari folder `dataset`, mengekstrak metrik RAM, waktu startup, switch kamera, inferensi, dan response time navigasi.

In [ ]:
def parse_log_file(filepath):
    with open(filepath, 'r') as f:
        content = f.read()
    
    data = {}
    
    # A. WAKTU STARTUP & RAM AWAL
    startup = re.search(r'Durasi App Startup\s*:\s*(\d+)\s*ms', content)
    data['startup'] = int(startup.group(1)) if startup else None
    
    ram_idle = re.search(r'Total RAM \(PSS\) awal\s*:\s*([\d\.]+)\s*MB', content)
    data['ram_idle'] = float(ram_idle.group(1)) if ram_idle else None
    
    # B. RESPONSE TIME SWITCH KAMERA
    switch_bf = re.search(r'Kamera Belakang -> Depan\s*:\s*(\d+)\s*ms', content)
    data['switch_back_front'] = int(switch_bf.group(1)) if switch_bf else None
    
    switch_fb = re.search(r'Kamera Depan -> Belakang\s*:\s*(\d+)\s*ms', content)
    data['switch_front_back'] = int(switch_fb.group(1)) if switch_fb else None
    
    # C. PENGUJIAN KINERJA INFERENSI
    # C.1 Metode A: Mock Bitmap
    c1_section = re.search(r'1\. METODE A: MOCK BITMAP IN-MEMORY.*?(?=2\. METODE B)', content, re.DOTALL)
    if c1_section:
        c1_content = c1_section.group(0)
        mock_avg = re.search(r'Rata-rata\s*:\s*([\d\.]+)\s*ms', c1_content)
        data['mock_inference'] = float(mock_avg.group(1)) if mock_avg else None
        
        ram_sebelum = re.search(r'RAM Sebelum.*?([\d\.]+)\s*MB\s*\(PSS\)', c1_content)
        data['ram_before_mock'] = float(ram_sebelum.group(1)) if ram_sebelum else None
        
        ram_setelah = re.search(r'RAM Setelah.*?([\d\.]+)\s*MB\s*\(PSS\)', c1_content)
        data['ram_after_mock'] = float(ram_setelah.group(1)) if ram_setelah else None
    
    # C.2 Metode B: Live Feed Camera
    c2_section = re.search(r'2\. METODE B: ALIRAN PREVIEW KAMERA LANGSUNG.*?(?=3\. PERBANDINGAN)', content, re.DOTALL)
    if c2_section:
        c2_content = c2_section.group(0)
        live_avg = re.search(r'Rata-rata\s*:\s*([\d\.]+)\s*ms', c2_content)
        data['live_inference'] = float(live_avg.group(1)) if live_avg else None
        
        ram_live = re.search(r'RAM Saat Live.*?([\d\.]+)\s*MB\s*\(PSS\)', c2_content)
        data['ram_live'] = float(ram_live.group(1)) if ram_live else None
        
    # D. TRANSISI NAVIGASI HALAMAN TENTANG
    d_section = re.search(r'D\. TRANSISI NAVIGASI HALAMAN TENTANG.*?(?=E\.)', content, re.DOTALL)
    if d_section:
        d_content = d_section.group(0)
        cam_about = re.search(r'Response Time \(Kamera -> Tentang\)\s*:\s*(\d+)\s*ms', d_content)
        about_cam = re.search(r'Response Time \(Tentang -> Kamera\)\s*:\s*(\d+)\s*ms', d_content)
        data['nav_cam_about'] = int(cam_about.group(1)) if cam_about else None
        data['nav_about_cam'] = int(about_cam.group(1)) if about_cam else None
        
    # E. TRANSISI NAVIGASI GALERI -> HALAMAN HASIL
    e_section = re.search(r'E\. TRANSISI NAVIGASI GALERI -> HALAMAN HASIL.*?(?=F\.)', content, re.DOTALL)
    if e_section:
        e_content = e_section.group(0)
        gal_res = re.search(r'Response Time \(Pilih Galeri -> Hasil\)\s*:\s*(\d+)\s*ms', e_content)
        res_cam_gal = re.search(r'Response Time \(Hasil -> Kamera Utama\)\s*:\s*(\d+)\s*ms', e_content)
        data['nav_gal_result'] = int(gal_res.group(1)) if gal_res else None
        data['nav_result_cam_gal'] = int(res_cam_gal.group(1)) if res_cam_gal else None
        
    # F. TRANSISI NAVIGASI CAPTURE (SHUTTER) -> HALAMAN HASIL
    f_section = re.search(r'F\. TRANSISI NAVIGASI CAPTURE.*?$', content, re.DOTALL)
    if f_section:
        f_content = f_section.group(0)
        cap_res = re.search(r'Response Time \(Shutter Capture -> Hasil\)\s*:\s*(\d+)\s*ms', f_content)
        res_cam_cap = re.search(r'Response Time \(Hasil -> Kamera Utama\)\s*:\s*(\d+)\s*ms', f_content)
        save_gal = re.search(r'Response Time Tombol Simpan Galeri\s*:\s*(\d+)\s*ms', f_content)
        data['nav_capture_result'] = int(cap_res.group(1)) if cap_res else None
        data['nav_result_cam_capture'] = int(res_cam_cap.group(1)) if res_cam_cap else None
        data['save_gallery'] = int(save_gal.group(1)) if save_gal else None
        
    # Mengumpulkan semua alokasi memori RAM PSS untuk mencari Peak RAM
    pss_values = []
    for m in re.finditer(r'Total RAM \\(PSS\\)(?:\\s*awal)?\\s*:\\s*([\\d\\.]+)\\s*MB', content):
        pss_values.append(float(m.group(1)))
    for m in re.finditer(r'([\\d\\.]+)\\s*MB\\s*\\(PSS\\)', content):
        pss_values.append(float(m.group(1)))
    data['ram_peak'] = max(pss_values) if pss_values else None
    
    return data

def load_device_data(dir_path):
    search_path = os.path.join('dataset', dir_path, "*.txt")
    files = glob.glob(search_path)
    temp_list = []
    for fp in files:
        with open(fp, 'r') as f:
            c = f.read()
        t_match = re.search(r'Tanggal Pengujian\s*:\s*(.*)', c)
        t_str = t_match.group(1).strip() if t_match else ""
        temp_list.append((t_str, fp))
    
    temp_list.sort()
    
    runs_data = []
    for i, (_, fp) in enumerate(temp_list, 1):
        parsed = parse_log_file(fp)
        parsed['Run'] = f"Pengujian {i}"
        runs_data.append(parsed)
    return pd.DataFrame(runs_data)

# Memuat data dari folder dataset
df_note10 = load_device_data('redmi note 10 pro')
df_redmi9a = load_device_data('redmi 9a')

print("Data Redmi Note 10 Pro:")
display(df_note10[['Run', 'startup', 'ram_idle', 'mock_inference', 'live_inference']])
print("\nData Redmi 9A:")
display(df_redmi9a[['Run', 'startup', 'ram_idle', 'mock_inference', 'live_inference']])

## A. Pengujian Penggunaan RAM

Pengujian penggunaan RAM dibagi menjadi tiga kondisi: Idle, Preview, dan Inference. Metrik RAM yang digunakan adalah PSS (Proportional Set Size) dalam satuan Megabytes (MB). Grafik di bawah ini membandingkan rata-rata penggunaan RAM antara kedua perangkat uji.

In [ ]:
avg_idle = df_note10['ram_idle'].mean()
avg_prev = df_note10['ram_live'].mean()
avg_inf = df_note10['ram_after_mock'].mean()

avg_idle_9a = df_redmi9a['ram_idle'].mean()
avg_prev_9a = df_redmi9a['ram_live'].mean()
avg_inf_9a = df_redmi9a['ram_after_mock'].mean()

# Data untuk Grafik RAM
categories = ['Idle', 'Preview', 'Inference']
means_note10 = [avg_idle, avg_prev, avg_inf]
means_redmi9a = [avg_idle_9a, avg_prev_9a, avg_inf_9a]

x = np.arange(len(categories))
width = 0.35  # Lebar batang

fig, ax = plt.subplots(figsize=(8, 6))
rects1 = ax.bar(x - width/2, means_note10, width, label='Redmi Note 10 Pro', color=COLORS[0], edgecolor='black', linewidth=0.7)
rects2 = ax.bar(x + width/2, means_redmi9a, width, label='Redmi 9A', color=COLORS[1], edgecolor='black', linewidth=0.7)

# Labeling
ax.set_ylabel('Penggunaan RAM (MB)', fontsize=12, fontweight='bold', labelpad=10)
ax.set_title('Gambar 4.35 Perbandingan Penggunaan RAM (PSS) pada Perangkat Uji', fontsize=13, fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(categories, fontsize=11, fontweight='bold')
ax.legend(frameon=True, facecolor='white', edgecolor='lightgrey')

# Batas Sumbu Y
ax.set_ylim(0, max(max(means_note10), max(means_redmi9a)) * 1.15)

def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.2f} MB',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=9, fontweight='semibold')

autolabel(rects1)
autolabel(rects2)

plt.tight_layout()
# Simpan grafik ke folder output
plt.savefig('output/Gambar_4.35_RAM.png', dpi=300)
plt.show()

## B. Pengujian Response Time

### Gambar 4.36 Perbandingan Response Time Aplikasi pada Perangkat Uji
Grafik di bawah ini membandingkan rata-rata response time navigasi transisi antar layar serta penyimpanan gambar ke galeri untuk kedua perangkat uji secara langsung.

In [ ]:
transitions = [
    ('Camera → About Screen', 'nav_cam_about'),
    ('About Screen → Camera', 'nav_about_cam'),
    ('Gallery → Result Screen', 'nav_gal_result'),
    ('Result Screen → Camera (via Gallery)', 'nav_result_cam_gal'),
    ('Capture → Result Screen', 'nav_capture_result'),
    ('Result Screen → Camera (via Capture)', 'nav_result_cam_capture'),
    ('Simpan ke Galeri', 'save_gallery')
]
keys_g5 = [t[1] for t in transitions]
labels_g5 = [t[0] for t in transitions]

means_g5_n10 = [df_note10[k].mean() for k in keys_g5]
means_g5_9a = [df_redmi9a[k].mean() for k in keys_g5]

y_g5 = np.arange(len(labels_g5))
height_g5 = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
rects1_g5 = ax.barh(y_g5 + height_g5/2, means_g5_n10, height_g5, label='Redmi Note 10 Pro', color=COLORS[0], edgecolor='black', linewidth=0.7)
rects2_g5 = ax.barh(y_g5 - height_g5/2, means_g5_9a, height_g5, label='Redmi 9A', color=COLORS[1], edgecolor='black', linewidth=0.7)

ax.set_xlabel('Response Time (ms)', fontsize=12, fontweight='bold', labelpad=10)
ax.set_title('Response Time Transisi Halaman & Simpan Gambar (Rata-rata)', fontsize=13, fontweight='bold', pad=15)
ax.set_yticks(y_g5)
ax.set_yticklabels(labels_g5, fontsize=10, fontweight='semibold')
ax.legend(frameon=True, facecolor='white', edgecolor='lightgrey')
ax.set_xlim(0, max(means_g5_9a) * 1.15)

for rect in rects1_g5:
    width_val = rect.get_width()
    ax.annotate(f'{width_val:.1f} ms',
                xy=(width_val, rect.get_y() + rect.get_height() / 2),
                xytext=(5, 0),
                textcoords="offset points",
                ha='left', va='center', fontsize=9, fontweight='semibold')

for rect in rects2_g5:
    width_val = rect.get_width()
    ax.annotate(f'{width_val:.1f} ms',
                xy=(width_val, rect.get_y() + rect.get_height() / 2),
                xytext=(5, 0),
                textcoords="offset points",
                ha='left', va='center', fontsize=9, fontweight='semibold')

plt.tight_layout()
# Simpan grafik ke folder output
plt.savefig('output/Gambar_4.36_Response_Time.png', dpi=300)
plt.show()

## C. Ringkasan & Perbandingan Perangkat

### Tabel 4.14 Perbandingan Performa Aplikasi Perangkat Uji
Tabel di bawah ini membandingkan spesifikasi perangkat keras (Sistem Operasi Android, Chipset, Kapasitas RAM) dengan rata-rata kinerja utama yang diukur (Waktu Startup, Waktu Inferensi Live vs Mock, Penggunaan RAM awal/Idle, dan Peak RAM saat aplikasi berjalan).

In [ ]:
startup_n10 = df_note10['startup'].mean() / 1000.0
startup_9a = df_redmi9a['startup'].mean() / 1000.0

inference_n10 = f"{df_note10['live_inference'].mean():.2f} ms (Live) / {df_note10['mock_inference'].mean():.2f} ms (Mock)"
inference_9a = f"{df_redmi9a['live_inference'].mean():.2f} ms (Live) / {df_redmi9a['mock_inference'].mean():.2f} ms (Mock)"

idle_ram_n10 = df_note10['ram_idle'].mean()
idle_ram_9a = df_redmi9a['ram_idle'].mean()

peak_ram_n10 = df_note10['ram_peak'].mean()
peak_ram_9a = df_redmi9a['ram_peak'].mean()

# Membuat DataFrame Tabel 4.14
tabel_perbandingan_hp = pd.DataFrame({
    'No': [1, 2],
    'Perangkat': ['Redmi Note 10 Pro', 'Redmi 9A'],
    'Android': ['Android 12 (API 31)', 'Android 10 (API 29)'],
    'Chipset': ['Snapdragon 732G', 'Helio G25'],
    'RAM': ['6 GB', '4 GB'],
    'Startup': [f"{startup_n10:.2f} s", f"{startup_9a:.2f} s"],
    'Inference': [inference_n10, inference_9a],
    'Idle RAM': [f"{idle_ram_n10:.2f} MB", f"{idle_ram_9a:.2f} MB"],
    'Peak RAM': [f"{peak_ram_n10:.2f} MB", f"{peak_ram_9a:.2f} MB"],
    'Status': ['Lancar', 'Lancar']
})

print("Tabel 4.14 Perbandingan Performa Aplikasi Perangkat Uji")
display(tabel_perbandingan_hp)

# Ekspor Tabel 4.14 ke folder output dalam format Markdown (.md) dan CSV (.csv)
with open('output/Tabel_4.14_Perbandingan_Performa.md', 'w') as f:
    f.write("# Tabel 4.14 Perbandingan Performa Aplikasi Perangkat Uji\n\n")
    f.write(tabel_perbandingan_hp.to_markdown(index=False))

tabel_perbandingan_hp.to_csv('output/Tabel_4.14_Perbandingan_Performa.csv', index=False)
print("\nTabel 4.14 berhasil diekspor ke folder output dalam format .md dan .csv!")

## D. Kesimpulan Analisis Kinerja

### Q&A
- **Bagaimana perbandingan performa antara Redmi Note 10 Pro dan Redmi 9A?**
  Redmi Note 10 Pro secara umum mengungguli Redmi 9A dalam kecepatan startup (3.44x lebih cepat) dan switch kamera. Namun, untuk inferensi deep learning dan navigasi halaman, perbedaannya tidak terlalu signifikan (hanya sekitar 1.05x - 1.28x lebih lambat pada Redmi 9A). Hal ini menunjukkan bahwa optimasi model deep learning pada aplikasi telah berjalan cukup baik bahkan untuk HP spesifikasi rendah.

### Data Analysis Key Findings
- **Durasi App Startup**: Redmi Note 10 Pro rata-rata membutuhkan waktu **3.53 detik** untuk startup, sedangkan Redmi 9A membutuhkan **12.15 detik** (3.44x lebih lama). Hal ini disebabkan oleh perbedaan chipset (Snapdragon 732G octa-core vs MediaTek Helio G25 octa-core) dan kecepatan baca media penyimpanan.
- **Konsumsi Memori RAM (PSS)**: Redmi Note 10 Pro mengonsumsi RAM PSS awal rata-rata sebesar **202.93 MB**, naik menjadi **211.89 MB** saat kamera preview aktif (Preview), dan stabil di sekitar **210.95 MB** setelah inferensi (Inference). Di sisi lain, Redmi 9A mengonsumsi memori PSS statis berturut-turut sebesar **146.28 MB** (Idle), **146.28 MB** (Preview), **146.28 MB** (Inference). Perbedaan ini menunjukkan pengelolaan alokasi memory heap di level OS Android 10 (Redmi 9A) yang membatasi pelaporan PSS atau melaporkannya secara statis dibandingkan Android 12 (Note 10 Pro).
- **Waktu Inferensi Model**: Pada Redmi Note 10 Pro, inferensi mock membutuhkan rata-rata **57.63 ms** dan live camera feed **69.10 ms**. Pada Redmi 9A, inferensi mock membutuhkan **73.97 ms** (hanya 1.28x lebih lama) dan live camera feed **73.33 ms** (hanya 1.06x lebih lama). Hasil ini sangat memuaskan karena menunjukkan model deep learning sangat portabel dan responsif pada kedua perangkat kelas menengah dan bawah.
- **Response Time Navigasi Halaman**: Transisi terlama adalah navigasi dari Galeri ke Halaman Hasil, yaitu rata-rata **1.99 detik** pada Note 10 Pro dan **2.23 detik** pada Redmi 9A, karena adanya proses decoding dan pemrosesan citra dari galeri sebelum membuka halaman hasil.

### Insights or Next Steps
- **Pengembangan Lanjutan**: Mengingat startup Redmi 9A memakan waktu hingga 12.15 detik, disarankan untuk menambahkan *asynchronous loading* atau *splash screen* yang dioptimalkan agar pengguna tidak merasa aplikasi membeku saat dijalankan.
- **Optimasi Navigasi Galeri**: Mengoptimalkan pembacaan gambar dari galeri dengan menerapkan downsampling gambar sebelum dikirim ke halaman hasil, untuk memangkas response time transisi Galeri → Hasil yang saat ini mendekati 2 detik.